# 13 · Modelling & Evaluation (2): 1-notch accuracy, RandomizedSearch, ensembles, final submission
Integrates the session *Modelling and Evaluation (2)*:

| Slide | Here |
|---|---|
| DSC metric: **1-notch accuracy** | `one_notch_accuracy()` = `within_one`; plus a decision rule that targets it |
| RandomizedSearchCV | Section 2: HistGradientBoosting tuned with random combinations, scored on 1-notch accuracy |
| "For loop" on preprocessing | notebook 10 (now also reports 1-notch accuracy) |
| Ensemble learning (stacking) | Section 3: soft voting and stacking with a meta-model trained on the **validation set** |
| Model understanding, SHAP | notebook 12 |
| Re-training on the whole DSC_training dataset | Section 5: final model refit on all 179k labelled buildings, then the scoring file |

Run after the model notebooks (at least 03, 05 and 08). Decisions are made on **validation**; **test** is reported once.

In [ ]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/dsc_utils.py) and import the shared helpers
root = Path.cwd()
while not (root / "src" / "dsc_utils.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dsc_utils import *

d = load_prepared()
x_train, x_val, x_test = d["x_train"], d["x_val"], d["x_test"]
y_train, y_val, y_test = d["y_train"], d["y_val"], d["y_test"]
feature_names = d["feature_names"]
print("train", x_train.shape, "| val", x_val.shape, "| test", x_test.shape)
import time, warnings
from scipy.stats import loguniform, randint
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
warnings.filterwarnings("ignore", category=UserWarning)

data = d["data"]
X_tr, X_va, X_te = data.iloc[d["indices_train"]], data.iloc[d["indices_val"]], data.iloc[d["indices_test"]]
cat_cols = data.select_dtypes(exclude="number").columns.tolist()
num_cols = data.select_dtypes(include="number").columns.tolist()

def hgb_pipe(**params):
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                               encoded_missing_value=-2), cat_cols),
        ("num", "passthrough", num_cols)])
    model = HistGradientBoostingClassifier(max_iter=500, early_stopping=True, random_state=42, **params)
    return Pipeline([("pre", pre), ("model", model)])

def X_for(name, split):
    """HistGradientBoosting models use the cleaned DataFrame, all others the preprocessed matrix."""
    if name.startswith("HGB"):
        return {"train": X_tr, "val": X_va, "test": X_te}[split]
    return {"train": x_train, "val": x_val, "test": x_test}[split]

class Precomputed:
    """Wraps already computed probabilities so evaluate() can score an ensemble."""
    def __init__(self, proba):
        self.proba = proba.reindex(columns=ORDER, fill_value=0)
        self.classes_ = np.array(ORDER)
    def predict_proba(self, X):
        return self.proba.to_numpy()
    def predict(self, X):
        return predict_one_notch(self.proba)

models = {n: load_model(n) for n in MODEL_NAMES + ["HGB_rs"] if (OUT / f"model_{n}.pkl").exists()}
print("Loaded models:", list(models))

## 1 · The DSC metric and a decision rule for it
**1-notch accuracy** counts a prediction as correct when it is at most one label away from the true label.

A model normally predicts its **most likely** label (argmax). For 1-notch accuracy it is better to pick the label whose
**neighbourhood** is most likely: P(label−1) + P(label) + P(label+1). Example: P(C) = 0.30, P(E) = 0.30, P(F) = 0.30.
Argmax may pick C, which is wrong for an F building; D or E covers two of the three likely labels.
The table compares both rules for every saved model. Decide on the **validation** columns.

In [ ]:
rows = []
for name, m in models.items():
    for split, y in [("val", y_val), ("test", y_test)]:
        proba = class_scores(m, X_for(name, split)).reindex(columns=ORDER, fill_value=0)
        rows.append({"model": name, "split": split,
                     "1-notch argmax": one_notch_accuracy(y, proba.idxmax(axis=1)),
                     "1-notch rule": one_notch_accuracy(y, predict_one_notch(proba)),
                     "exact argmax": float((proba.idxmax(axis=1).to_numpy() == y).mean())})
rule_table = pd.DataFrame(rows).pivot(index="model", columns="split").round(4)
rule_table

## 2 · RandomizedSearchCV for HistGradientBoosting
Instead of trying every combination (GridSearchCV), each iteration draws **one random combination** from the given
ranges. 12 iterations × 3 folds on a 30k training sample, scored directly on 1-notch accuracy with the decision rule.
Takes roughly 10–30 minutes on a laptop; lower `n_iter` to go faster.

In [ ]:
xs, ys = stratified_sample(X_tr, y_train, 30000)
param_dist = {
    "model__learning_rate": loguniform(0.02, 0.3),
    "model__max_leaf_nodes": randint(15, 256),
    "model__min_samples_leaf": randint(10, 200),
    "model__l2_regularization": loguniform(1e-3, 10),
}
t = time.time()
rs = RandomizedSearchCV(hgb_pipe(), param_dist, n_iter=12, scoring=one_notch_scorer("window"),
                        cv=StratifiedKFold(3, shuffle=True, random_state=0), random_state=0, n_jobs=2, verbose=1)
rs.fit(xs, ys)
print(f"done in {time.time() - t:.0f}s | best CV 1-notch accuracy: {rs.best_score_:.4f}")
print(rs.best_params_)
pd.DataFrame(rs.cv_results_)[["params", "mean_test_score", "std_test_score"]].sort_values("mean_test_score", ascending=False).head(5)

In [ ]:
# refit the best combination on the FULL training set, then evaluate once on test (with the 1-notch rule)
HGB_rs = hgb_pipe().set_params(**rs.best_params_).fit(X_tr, y_train)
print("validation 1-notch:", round(one_notch_accuracy(y_val, OneNotchClassifier(HGB_rs).predict(X_va)), 4))
evaluate("HGB RandomizedSearch (1-notch rule)", OneNotchClassifier(HGB_rs), X_te, y_test)
save_model(HGB_rs, "HGB_rs")
models["HGB_rs"] = HGB_rs

## 3 · Ensemble learning
Base models are already trained on the **training** set. Their class probabilities on the **validation** set are the
input for the ensembles, so nothing is fitted on test.
- **Soft voting:** average the probabilities of several models.
- **Stacking (slides):** the probability vectors S(x) of all base models become the features of a **meta-model**
  (logistic regression), trained on a separate dataset, here the validation set. Its validation score is estimated with
  5-fold cross-validation **inside** the validation set.

SVMs are left out: they give no real probabilities.

In [ ]:
prob_models = [n for n, m in models.items() if hasattr(m, "predict_proba")]
P_val = {n: class_scores(models[n], X_for(n, "val")).reindex(columns=ORDER, fill_value=0) for n in prob_models}
P_te = {n: class_scores(models[n], X_for(n, "test")).reindex(columns=ORDER, fill_value=0) for n in prob_models}

single = pd.Series({n: one_notch_accuracy(y_val, predict_one_notch(P_val[n])) for n in prob_models}).sort_values(ascending=False)
print("Validation 1-notch accuracy per base model (with the rule):")
print(single.round(4))
top3 = single.index[:3].tolist()
print("Top 3:", top3)

In [ ]:
def average(P, names):
    return sum(P[n] for n in names) / len(names)

voting = {}
for label, names in [("vote_all", prob_models), ("vote_top3", top3)]:
    voting[label] = names
    print(f"{label:10s} {names} | validation 1-notch: {one_notch_accuracy(y_val, predict_one_notch(average(P_val, names))):.4f}")

In [ ]:
# Stacking: meta-model on the base models' validation probabilities
S_val = np.hstack([P_val[n].to_numpy() for n in prob_models])
S_te = np.hstack([P_te[n].to_numpy() for n in prob_models])
meta = LogisticRegression(C=1.0, max_iter=2000)

oof = cross_val_predict(meta, S_val, y_val, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")
oof = pd.DataFrame(oof, columns=np.unique(y_val))
print("Stacking, validation 1-notch (5-fold CV inside validation):", round(one_notch_accuracy(y_val, predict_one_notch(oof)), 4))

meta.fit(S_val, y_val)
P_stack_te = pd.DataFrame(meta.predict_proba(S_te), columns=meta.classes_)

Test scores of the ensembles (reported once, added to the results table):

In [ ]:
evaluate("Soft voting (all)", Precomputed(average(P_te, voting["vote_all"])), None, y_test, plot=False)
evaluate("Soft voting (top 3)", Precomputed(average(P_te, voting["vote_top3"])), None, y_test, plot=False)
evaluate("Stacking (meta LR)", Precomputed(P_stack_te), None, y_test)

## 4 · Choose the final model
The `within_one` column **is** the DSC's 1-notch accuracy. Choose on validation (sections 1–3); the test column confirms it.

In [ ]:
results = pd.read_csv(RESULTS_FILE).drop_duplicates("model", keep="last")
results.sort_values("within_one", ascending=False).round(4)

In [ ]:
# FINAL = list of base models whose probabilities are averaged (one name = a single model)
# e.g. ["HGB_rs"], or top3 for soft voting. Stacking is not refitted here (its meta-model needs a held-out set).
FINAL = ["HGB_rs"]
print("Final model:", FINAL)

## 5 · Re-train on the whole DSC_training dataset and predict the scoring file
The split was only for choosing. The final model is refitted on **all 179k labelled buildings** (train + validation +
test) with the chosen hyperparameters, then predicts the scoring file with the 1-notch decision rule.
If notebook 02 added neighbourhood-label features, they are recomputed here from all labelled buildings
(out-of-fold for the labelled rows, so no building sees its own label).

In [ ]:
n = len(data)
y_all = np.empty(n, dtype=object)
for idx, yy in [(d["indices_train"], y_train), (d["indices_val"], y_val), (d["indices_test"], y_test)]:
    y_all[idx] = yy
data_all, score_all = data.copy(), d["score_data"].copy()

if "nb_mean_label" in data_all.columns:
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.model_selection import KFold
    coords = np.radians(data_all[["y_ll", "x_ll"]].to_numpy())
    score_coords = np.radians(score_all[["y_ll", "x_ll"]].to_numpy())
    targets = [pd.Series(y_all).map(TO_I).to_numpy().astype(float), np.isin(y_all, POOR).astype(float)]
    def nb_feats(fit_idx, query):
        return np.column_stack([KNeighborsRegressor(n_neighbors=20, metric="haversine", algorithm="ball_tree")
                                .fit(coords[fit_idx], t[fit_idx]).predict(query) for t in targets])
    nb = np.zeros((n, 2))
    for fit_part, pred_part in KFold(5, shuffle=True, random_state=0).split(np.arange(n)):
        nb[pred_part] = nb_feats(fit_part, coords[pred_part])
    data_all["nb_mean_label"], data_all["nb_share_EFG"] = nb[:, 0], nb[:, 1]
    nb_s = nb_feats(np.arange(n), score_coords)
    score_all["nb_mean_label"], score_all["nb_share_EFG"] = nb_s[:, 0], nb_s[:, 1]
    print("neighbourhood features recomputed on all labelled rows")

probas, X_all_matrix = [], None
for name in FINAL:
    t = time.time()
    model = clone(models[name])
    if name.startswith("HGB"):
        model.fit(data_all, y_all)
        probas.append(class_scores(model, score_all))
    else:
        if X_all_matrix is None:
            pre_all = clone(d["preprocessor"])
            X_all_matrix = pre_all.fit_transform(data_all)
            X_score_matrix = pre_all.transform(score_all)
        model.fit(X_all_matrix, y_all)
        probas.append(class_scores(model, X_score_matrix))
    save_model(model, f"FINAL_{name}")
    print(f"refitted {name} on {n} buildings in {time.time() - t:.0f}s")

final_proba = sum(p.reindex(columns=ORDER, fill_value=0) for p in probas) / len(probas)
submission = pd.DataFrame({"finc_id": d["score_ids"], "EPC label": predict_one_notch(final_proba)})
print(pd.DataFrame({"training": pd.Series(y_all).value_counts(normalize=True),
                    "predicted": submission["EPC label"].value_counts(normalize=True)}).reindex(ORDER).round(3))

In [ ]:
out = OUT / "predictions_group22_final.csv"          # adapt to the organisers' template before submitting
submission.to_csv(out, index=False)
print("Saved", out, submission.shape)